# Seen and Unseen — Experiment 3: record the freeze environment (before the freeze)

Run this once on the same kind of Colab runtime as the confirmation (A100, High-RAM). It installs the packages the confirmation uses and writes the **environment block** for the freeze manifest: the hash of the repository lockfile, the exact versions of Python, torch, CUDA, numpy, scipy, h5py, transformers and accelerate with their hash, and the hardware class. The locked confirmation notebook installs these same versions, and the runner refuses a run on other hardware or versions unless a logged deviation is given.

Takes about 2 minutes. Downloads `environment.json`; paste it into the freeze manifest as `"environment"`.

In [ ]:
#@title 0. Configuration
HARDWARE_CLASS = "A100"   #@param {type:"string"}
CODE_COMMIT = "6b1031b10ac5ec757e8bd8e7f1a2b47f0ddcbfef"  #@param {type:"string"}   # the commit the freeze tag will point to
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (switch to a GPU runtime)")

In [ ]:
#@title 1. Code and packages (as the confirmation installs them)
%cd /content
import os
if not os.path.exists("trajectory-lineage"):
    !git clone -q https://github.com/physicalreasoning/trajectory-lineage.git
%cd /content/trajectory-lineage
!git fetch -q && git checkout -q {CODE_COMMIT} && git rev-parse --short HEAD
!pip install -q numpy h5py "transformers>=5.0" accelerate

In [ ]:
#@title 2. Write and download the environment block
%cd /content/trajectory-lineage
!python scripts/freeze.py environment --hardware-class "{HARDWARE_CLASS}" --out /content/environment.json
from google.colab import files
files.download("/content/environment.json")